# 多行 TRL

多线 TRL 是一种双端口 VNA 校准方法，它利用至少两条具有不同物理长度的传输线和至少一个在两个端口上都相同的反射校准标准。不需要知道这些线路的电参数，但传输线应具有相同的结构（相同的传播常数和特性阻抗）。不需要精确知道反射校准标准的反射系数，但需要以 90 度的精度知道相位。如果测得的线路相位差是 180 度的倍数，则校准结果将是奇异的。校准精度会随着线路测量相位接近奇异点而降低，当两个线路的相位差为 90 度时，可获得最佳精度。可以使用多个线路来扩展校准精度有效的频率范围。

本示例演示如何使用 `skrf` 的 NIST 风格多线校准 (`NISTMultilineTRL`)。首先，介绍一个简单的应用示例，然后通过一个完整的仿真，展示校准准确性与使用的线路数量之间的关系。所有用于演示的数据均由 `skrf` 生成，相应的代码位于示例的末尾。

此外，本示例展示了 TUG 多线校准 (TUGMultilineTRL)，它是一种替代常用 NIST 风格多线方法的方案。TUG 方法具有更好的统计特性。示例中还提供了通过 [蒙特卡洛分析](#monte-carlo-analysis---nist-vs-tug) 对比这两种校准方法的案例。

## 简单的多线 TRL

### 设置

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

import skrf
from skrf.calibration import NISTMultilineTRL, TUGMultilineTRL, terminate
from skrf.media import CPW, Coaxial
from skrf.network import two_port_reflect

skrf.stylely()

### 将数据加载到 skrf 中

In [ ]:
# Load all measurement data into a dictionary
data = skrf.io.general.read_all_networks('multiline_trl_data/')

# Pull out measurements by name into an ordered list
measured_names = ['thru','reflect','linep3mm','line2p3mm']
measured = [data[k] for k in measured_names]

# Switch terms
gamma_f,gamma_r = data['gamma_f'],data['gamma_r']

# DUT
dut_meas = data['DUT']

# 50 ohm termination
res_50ohm_meas = data['res_50ohm']

### 简单的多线 TRL

In [ ]:
# define the line lengths in meters (including thru)
l = [0, 0.3e-3, 2.3e-3]

# Do the calibration
cal = NISTMultilineTRL(
    measured = measured,  # Measured standards
    Grefls = [-1], # Reflection coefficient of the reflect, -1 for short
    l = l,         # Lengths of the lines
    er_est = 7,    # Estimate of transmission line effective permittivity
    switch_terms = (gamma_f, gamma_r) # Switch terms
    )

# Correct the DUT using the above calibration
corrected = cal.apply_cal(dut_meas)

corrected.plot_s_db()

## 比较不同线路组合下的校准结果

在这里，我们将循环遍历不同的线路组合，以展示校准精度的差异。

In [ ]:
# Run NIST Multiline TRL calibration with different combinations of lines

# Put through and reflect to their own list ...
mtr = measured[:2]

# and lines on their own
mlines = measured[2:]

line_len = l[1:]

cals = []
duts = []

line_combinations = [[0], [1], [0,1]]

for used_lines in line_combinations:

    m = mtr + [mlines[i] for i in used_lines]

    # Add thru length to list of line lengths
    l = [l[0]] + [line_len[i] for i in used_lines]

    # Do the calibration
    cal = NISTMultilineTRL(
        measured = m,  # Measured standards
        Grefls = [-1], # Reflection coefficient of the reflect, -1 for short
        l = l,         # Lengths of the lines
        er_est = 7,    # Estimate of transmission line effective permittivity
        switch_terms = (gamma_f, gamma_r) # Switch terms
        )

    # Correct the DUT using the above calibration
    corrected = cal.apply_cal(dut_meas)
    corrected.name = f'DUT, lines {used_lines}'

    duts.append(corrected)
    cals.append(cal)

### 校正后的DUT的传输特性绘制使用不同校准线的组合进行校准后的DUT的特性曲线。

In [ ]:
plt.figure()
plt.title('DUT S21')
for dut in duts:
    dut.plot_s_db(m=1, n=0)

### 校正后的DUT的S11，使用不同数量的校准线S11显示出更大的变化。* 使用一条短路线时，低频段的噪声很大。* 仅使用长线时，在直通线和校准线的相位差接近180度的倍数时，校准精度非常差。* 使用两条线时，校准精度在所有频率下都很好。

In [ ]:
plt.figure()
plt.title('DUT S11')
for dut in duts:
    dut.plot_s_db(m=0, n=0)

### 不同校准方法的归一化标准差可以使用归一化标准差来衡量校准的准确性。数值越小，表示校准对测量噪声的敏感度越低。* 采用一条 90 度长的传输线进行 TRL 校准时，归一化标准差为 1。* 采用一条 180 度长的无损耗传输线进行 TRL 校准时，结果为奇异解，归一化标准差为无穷大。* 采用多条传输线进行校准时，归一化标准差可以小于 1。请注意，归一化标准差 (nstd) 是经过归一化处理的，因此它不考虑实际的测量噪声。它仅根据求解出的传播常数和传输线长度进行计算。允许的最大阈值取决于被测器件、测量噪声以及所需的测量精度。如果存在较大的尖峰，例如在下面的长传输线案例中可见，这表明在特定频率下，校准结果非常接近于奇异解，并且测量精度会很差。

In [ ]:
f_ghz = dut.frequency.f_scaled

plt.figure()
plt.title('Calibration normalized standard deviation')
for e, cal in enumerate(cals):
    plt.plot(f_ghz, cal.nstd, label=f'Lines: {line_combinations[e]}')
plt.ylim([0,20])
plt.legend(loc='upper right')
dut.frequency.labelXAxis()

## 计算校准中使用的传输线的有效复相对介电常数传输线的有效复相对介电常数 $\epsilon_{r,eff}$ 与传播常数 $\gamma$ 存在以下关系：$\gamma = \frac{2\pi f}{c}\sqrt{\epsilon_{r,eff}}$，其中 $c$ 等于光速，$f$ 为频率。通常，它是一个复数值，其虚部表示损耗。

In [ ]:
# Define calibration standard media
freq = dut.frequency

# Get the cal with the both lines
cal = cals[-1]

plt.figure()
plt.title('CPW effective permittivity (real part)')
plt.plot(f_ghz, cal.er_eff.real, label='Solved er_eff')
plt.xlabel('Frequency (GHz)')
plt.legend(loc='lower right')

当传输线长度差为 90 度时，TRL 校准的精度最高。然而，求解的传播常数和有效介电常数时，传输线长度差越大，结果越准确。在低频时，由于传输线相位差较小，因此估计值会产生更多噪声。

## 绘制已求解反射系数的相位图

将校准应用于测量的反射校准标准，我们可以得到未知反射器的校准后的散射参数。

In [ ]:
plt.figure()
plt.title('Solved reflection coefficient of the reflect standard')
cal.apply_cal(measured[1]).plot_s_deg(n=0, m=0, label='Solved short')

## Reference plane shiftBecause propagation constant of the media is solved during the calibration it's possible to shift the reference plane by a specified distance.The reference plane shift can be specified with `ref_plane` argument. The shift should be specified in meters, negative lengths is towards the VNA. By default the same shift is applied to both ports. Unequal shift on the two ports is supported by passing a two element list.

In [ ]:
cal_shift = NISTMultilineTRL(
    measured = measured,  # Measured standards
    Grefls = [-1], # Reflection coefficient of the reflect, -1 for short
    l = l,         # Lengths of the lines
    er_est = 7,    # Estimate of transmission line effective permittivity
    switch_terms = (gamma_f, gamma_r), # Switch terms
    # Shift reference planes twords VNA by this amount (in m) on both ports
    ref_plane = -50e-6
    )

# Correct the DUT using the above calibration
corrected_thru = cal.apply_cal(measured[0])
corrected_thru_shifted = cal_shift.apply_cal(measured[0])

corrected_thru.plot_s_deg(m=1, n=0, label='Thru phase')
corrected_thru_shifted.plot_s_deg(m=1, n=0, label='Reference plane shifted thru phase')

## 校准参考阻抗重整化校准的参考阻抗默认值为传输线的特性阻抗。如果已知线路的实际特性阻抗，我们可以将其作为 `z0_line` 参数传递给校准程序，以将测量的散射参数重整化到固定的参考阻抗 `z0_ref`。如果单位长度的电导 (G) 远小于单位长度的容抗 ($j\omega C_0$)，则传输线的特性阻抗可以表示为传播常数 $\gamma$ 和单位长度的电容 $C_0$：$Z_0 = \gamma/(j 2 \pi f C_0)$如果已知 $C_0$，可以将它作为 `c0` 参数传递给校准程序，以将校准参考阻抗重整化为 `z0_ref`（默认值为 50 欧姆），假设 G = 0。如果线路有损耗，则特性阻抗为复数值，在这种情况下，提供单个 `c0` 值而不是固定的 `z0_line` 通常更准确。在这种情况下，我们知道线路的特性阻抗实际上为 55 欧姆。要将校准从 55 欧姆重整化到 50 欧姆，我们需要将 `z0_line=55` 参数传递给校准程序。

In [ ]:
cal_ref = NISTMultilineTRL(
    measured = measured,  # Measured standards
    Grefls = [-1], # Reflection coefficient of the reflect, -1 for short
    l = l,         # Lengths of the lines
    er_est = 7,    # Estimate of transmission line effective permittivity
    switch_terms = (gamma_f, gamma_r), # Switch terms
    z0_line = 55, # Line actual characteristic impedance
    z0_ref = 50 # Calibration reference impedance
    )

cal.apply_cal(res_50ohm_meas).s11.plot_s_db(label=r'50 $\Omega$ termination |$S_{11}$|, Z_ref = line')
cal_ref.apply_cal(res_50ohm_meas).s11.plot_s_db(label=r'50 $\Omega$ termination |$S_{11}$|, Z_ref = 50 $\Omega$')

重新标准化后，50 欧姆终端的测量结果显示出良好的匹配。由于测量中的噪声，匹配结果并不完全理想。

## TUG 多线 TRLTUG 多线 TRL (`TUGMultilineTRL`) 是一种实现多线 TRL 校准的方法，其方法与 NIST 的方法不同。TUG 多线中校准标准的定义与 NIST 多线中的定义相同，需要至少两条线，并且是对称反射。关于 TUG 多线的更多数学细节，请参见此处：https://ziadhatab.github.io/posts/multiline-trl-calibration/以下是一个比较 NIST 多线和 TUG 多线的示例。两种方法都得到相同的结果。但是，在后面的 [蒙特卡罗 (MC) 分析](#monte-carlo-analysis---nist-vs-tug) 中，我们将看到 TUG 方法与 NIST 方法相比，具有更好的统计性能。

In [ ]:
# Load all measurement data into a dictionary
data = skrf.io.general.read_all_networks('multiline_trl_data/')
# Pull out measurements by name into an ordered list
measured_names = ['thru','reflect','linep3mm','line2p3mm']
measured = [data[k] for k in measured_names]
# Switch terms
gamma_f,gamma_r = data['gamma_f'],data['gamma_r']
# DUT
dut_meas = data['DUT']
# define the line lengths in meters (including thru)
l = [0, 0.3e-3, 2.3e-3]

# NIST multiline TRL
cal = NISTMultilineTRL(
    measured = measured,  # Measured standards
    Grefls = [-1], # Reflection coefficient of the reflect, -1 for short
    l = l,         # Lengths of the lines
    er_est = 7,    # Estimate of transmission line effective permittivity
    switch_terms = (gamma_f, gamma_r) # Switch terms
    )

# Correct the DUT using the above calibration
corrected = cal.apply_cal(dut_meas)
corrected.name = 'DUT NIST'

corrected.plot_s_db()

# TUG multiline TRL - note the difference in
cal = TUGMultilineTRL(
        line_meas=[measured[0]]+measured[2:],
        line_lengths=l,
        er_est=7,
        reflect_meas=[measured[1]],
        reflect_est=[-1],
        switch_terms = (gamma_f, gamma_r)
        )

# Correct the DUT using the above calibration
corrected = cal.apply_cal(dut_meas)
corrected.name = 'DUT TUG'

corrected.plot_s_db(linestyle='--')

### 无反射测量时的校准`TUGMultilineTRL` 的一个优点是，如果您只需要校准后的DUT的传播常数以及S21和S12参数，则可以使用单独的传输线校准标准。但是，要获得校准后的DUT的正确S11和S22值，您还需要提供反射校准标准的测量。这可以在下面的示例中看到，在使用和不使用反射校准标准时，S11和S22参数会产生偏差。

In [ ]:
# Full TUG multiline TRL
cal = TUGMultilineTRL(
        line_meas=[measured[0]]+measured[2:],
        line_lengths=l,
        er_est=7,
        reflect_meas=[measured[1]],
        reflect_est=[-1],
        switch_terms = (gamma_f, gamma_r)
        )

# Correct the DUT using the above calibration
corrected = cal.apply_cal(dut_meas)
corrected.name = 'TUG w/ reflect'

corrected.plot_s_db(linestyle='-')

# TUG multiline without reflect measurement
cal = TUGMultilineTRL(
        line_meas=[measured[0]]+measured[2:],
        line_lengths=l,
        er_est=7,
        switch_terms = (gamma_f, gamma_r)
        )

# Correct the DUT using the above calibration
corrected = cal.apply_cal(dut_meas)
corrected.name = 'TUG w/o reflect'

corrected.plot_s_db(linestyle='--')

### 校准特征值NIST 方法采用归一化的标准差，以强调校准的灵敏度。类似地，TUG 方法使用一种基于校准问题的特征值的指标。TUG 多线方法仅求解一个加权特征值分解，结果得到一个实数值的特征值 $\lambda$。这个特征值表示校准的稳定性。特征值越接近零，校准的灵敏度就越高。本质上，它可以被视为 NIST 方法中定义的归一化标准差的反向指标。

In [ ]:
f_ghz = dut.frequency.f_scaled

plt.figure()
plt.title('TUG multiline eigenvalue')
plt.plot(f_ghz, cal.lambd)
dut.frequency.labelXAxis()

## 蒙特卡洛分析 - NIST 与 TUG使用蒙特卡洛分析，我们生成包含随机噪声的校准标准的合成数据，并多次重复该过程，每次都使用新的噪声样本。在每次试验中，我们还会计算并存储校准后的 DUT，并使用它来计算校准方法的平均绝对误差 (MAE)。MAE 的一般定义如下：$$\text{MAE} = \frac{1}{N}\sum_{i=1}^{N} |x_i - x_\mathrm{true}|$$其中，$x_i$ 是第 $i$ 次蒙特卡洛试验的结果，$x_\mathrm{true}$ 是真实值。以下代码生成参考真实值。

In [ ]:

freq = skrf.Frequency(1, 100, 201, 'GHz')
# CPW media used for DUT and the calibration standards
cpw = CPW(freq, w=40e-6, s=51e-6, ep_r=12.9, t=5e-6, rho=2e-8)
# 1.0 mm coaxial media for calibration error boxes
coax1mm = Coaxial(freq, z0_port=50, Dint=0.434e-3, Dout=1.0e-3, sigma=1e8)
f_ghz = cpw.frequency.f*1e-9
# Error boxes
X = coax1mm.line(1, 'm', z0=58, name='X')
Y = coax1mm.line(1.1, 'm', z0=40, name='Y')
# Realistic looking switch terms
gamma_f = coax1mm.delay_load(0.2, 21e-3, 'm', z0=60)
gamma_r = coax1mm.delay_load(0.25, 16e-3, 'm', z0=56)
# Lengths of the lines used in the calibration, units are in meters
line_lengths = [0, 0.3e-3, 2.3e-3]  # first is thru
lines = [cpw.line(l, 'm') for l in line_lengths]
# Attenuator with mismatched feed lines
dut_feed = cpw.line(1.5e-3, 'm', z0=60)
dut = dut_feed**cpw.attenuator(-10)**dut_feed
# Reflect standard
reflect_offset = 10e-6
short = cpw.delay_short(reflect_offset, 'm')
short = two_port_reflect(short, short)

# Measured... embedded in the error boxes
line_meas = [terminate(X**k**Y,gamma_f, gamma_r) for k in lines]
reflect_meas = [terminate(X**short**Y, gamma_f, gamma_r)]
dut_meas = terminate(X**dut**Y, gamma_f, gamma_r)
er_est = 6.3-0.0001j
reflect_est = [-1]

# NIST multiline TRL (Noiseless case as reference)
measured = [line_meas[0]] + [reflect_meas[0]] + line_meas[1:]
cal = NISTMultilineTRL(
    measured = measured,
    Grefls = reflect_est,
    l = line_lengths,
    refl_offset = reflect_offset,
    er_est = er_est,
    switch_terms=[gamma_f, gamma_r])

dut_ideal = cal.apply_cal(dut_meas)
dut_ideal.name = 'DUT'
dut_ideal.plot_s_db()

现在，我们开始进行蒙特卡洛分析，方法是在极坐标中引入噪声。

In [ ]:
# Monte Carlo Analysis
print('MC starts: ')
M = 100 # number of trials
mag_std = 0.05 # std of magnitude (linear)
phase_std = 20 # std of phase (degrees)

cal_dut_NIST = []
cal_dut_TUG  = []

for inx in range(M):
    # add noise (make sure to copy and not override the original data)
    lines_n   = [NW.copy() for NW in line_meas]
    for NW in lines_n:
        NW.add_noise_polar(mag_std, phase_std)

    reflect_n   = [NW.copy() for NW in reflect_meas]
    for NW in reflect_n:
        NW.add_noise_polar(mag_std, phase_std)

    # TUG multiline TRL
    cal = TUGMultilineTRL(line_meas=lines_n, line_lengths=line_lengths, er_est=er_est,
                reflect_meas=reflect_n, reflect_est=reflect_est, reflect_offset=reflect_offset,
                switch_terms=[gamma_f, gamma_r])
    cal_dut_TUG.append(cal.apply_cal(dut_meas))

    # NIST multiline TRL
    measured = [lines_n[0]] + [reflect_n[0]] + lines_n[1:]
    cal = NISTMultilineTRL(
        measured = measured,
        Grefls = reflect_est,
        l = line_lengths,
        refl_offset = reflect_offset,
        er_est = er_est,
        switch_terms=[gamma_f, gamma_r])
    cal_dut_NIST.append(cal.apply_cal(dut_meas))
    print(f'MC Index: {inx+1}/{M} done.', end='\r', flush=True)


最后，我们计算并绘制平均绝对误差（MAE）。结果表明，与 NIST 方法相比，TUG 多线方法具有更低的误差。

In [ ]:
def dut_MAE(MC, ideal):
    # compute mean absolute error
    return np.array([ abs(x.s-ideal.s) for x in MC ]).mean(axis=0)

MAE_NIST = dut_MAE(cal_dut_NIST, dut_ideal)
MAE_TUG  = dut_MAE(cal_dut_TUG, dut_ideal)

f_ghz = dut_ideal.frequency.f_scaled

plt.figure()
plt.title('Mean Absolute Error')
plt.plot(f_ghz, MAE_NIST[:,0,0], label='NIST, S11')
plt.plot(f_ghz, MAE_NIST[:,1,0], label='NIST, S21')
plt.plot(f_ghz, MAE_TUG[:,0,0], label='TUG, S11')
plt.plot(f_ghz, MAE_TUG[:,1,0], label='TUG, S21')
plt.legend(loc='upper right')
dut_ideal.frequency.labelXAxis()

## 模拟生成输入数据

以下是生成上述数据的方法。

### 创建频率和介质

In [ ]:
freq = skrf.Frequency(1, 100, 201, 'GHz')

# CPW media used for DUT and the calibration standards
cpw = CPW(freq, w=40e-6, s=51e-6, ep_r=12.9,
                     t=5e-6, rho=2e-8)
print(cpw.z0[0])

# 1.0 mm coaxial media for calibration error boxes
coax1mm = Coaxial(freq, z0_port=50, Dint=0.434e-3, Dout=1.0e-3, sigma=1e8)
print(coax1mm.z0[0])

f_ghz = cpw.frequency.f*1e-9

### 创建逼真的误差网络。传播常数的确定是一个迭代过程，当误差网络随机生成时，其效果会降低。

In [ ]:
X = coax1mm.line(1, 'm', z0=58, name='X')
Y = coax1mm.line(1.1, 'm', z0=40, name='Y')

plt.figure()
plt.title('Error networks')
X.plot_s_db()
Y.plot_s_db()

# Realistic looking switch terms
gamma_f = coax1mm.delay_load(0.2, 21e-3, 'm', z0=60)
gamma_r = coax1mm.delay_load(0.25, 16e-3, 'm', z0=56)

plt.figure()
plt.title('Switch terms')
gamma_f.plot_s_db()
gamma_r.plot_s_db()

### 生成虚拟测量数据

In [ ]:
# Lengths of the lines used in the calibration, units are in meters
line_len = [0.3e-3, 2.3e-3]
lines = [cpw.line(l, 'm') for l in line_len]

# Attenuator with mismatched feed lines
dut_feed = cpw.line(1.5e-3, 'm', z0=60)
dut = dut_feed**cpw.attenuator(-10)**dut_feed

res_50ohm = cpw.resistor(50) ** cpw.short(nports=2) ** cpw.resistor(50)

# Through and non-ideal short
# Real reflection coefficient is solved during the calibration

short = cpw.delay_short(10e-6, 'm')

actuals = [
    cpw.thru(),
    two_port_reflect(short, short),
    ]

actuals.extend(lines)

# Measured
measured = [X**k**Y for k in actuals]

# Switch termination
measured = [terminate(m, gamma_f, gamma_r) for m in measured]

# Add little noise to the measurements
for m in measured:
    m.add_noise_polar(0.001, 0.1)

names = ['thru', 'reflect', 'linep3mm', 'line2p3mm']
for k,name in enumerate(names):
    measured[k].name=name


# Noiseless DUT so that all the noise will be from the calibration
dut_meas = terminate(X**dut**Y, gamma_f, gamma_r)
dut_meas.name = 'DUT'

res_50ohm_meas = terminate(X**res_50ohm**Y, gamma_f, gamma_r)
res_50ohm_meas.name = 'res_50ohm'

# Put through and reflect to their own list ...
mtr = measured[:2]

# and lines on their own
mlines = measured[2:]

# write data to disk
write_data = False
if write_data:
    [k.write_touchstone(dir='multiline_trl_data/') for k in measured]
    gamma_f.write_touchstone('multiline_trl_data/gamma_f.s1p')
    gamma_r.write_touchstone('multiline_trl_data/gamma_r.s1p')
    dut_meas.write_touchstone(dir='multiline_trl_data/')
    res_50ohm_meas.write_touchstone(dir='multiline_trl_data/')